# Kata 13 — Code Review Headless en CI/CD

> Spec: [`specs/013-headless-ci-review/spec.md`](../../specs/013-headless-ci-review/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

# Sonnet para mayor matiz en review
client, settings = bootstrap(model="claude-sonnet-4-6", budget_calls=6)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Un reviewer determinista corre en CI con `claude -p` (CLI) o vía API directa. Produce **JSON estructurado** validado contra schema. El pipeline parsea con `json.loads` + schema validator y publica anotaciones en el PR. Cero regex sobre prosa.

## 2. Por qué importa

Cuando el modelo cambie de redacción (idioma, formato, longitud), el regex que parsea prosa romperá. El schema sobrevive cualquier reformulación.

## 3. Ejemplo correcto — `tool_choice` forzado + schema

In [ ]:
REVIEW_TOOL = {
    "name": "post_review",
    "description": "Publica los hallazgos de la revisión.",
    "input_schema": {
        "type": "object",
        "properties": {
            "annotations": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "file": {"type": "string"},
                        "line": {"type": "integer", "minimum": 1},
                        "severity": {"type": "string", "enum": ["info","warning","error"]},
                        "rule_id": {"type": "string"},
                        "message": {"type": "string"},
                    },
                    "required": ["file","line","severity","rule_id","message"],
                },
            },
            "summary": {"type": "string"},
        },
        "required": ["annotations","summary"],
    },
}

DIFF = '''diff --git a/src/auth.py b/src/auth.py
@@ -10,3 +10,7 @@
 def login(user, pwd):
+    if pwd == "admin123":
+        return True
     return user.password == pwd
'''

system = (
    "Eres un reviewer de seguridad. Analiza el diff y publica anotaciones tipadas. "
    "Detecta credenciales hardcodeadas, comparaciones inseguras y secrets."
)
resp = client.messages.create(
    system=system,
    messages=[{"role":"user","content": f"```diff\n{DIFF}\n```"}],
    tools=[REVIEW_TOOL],
    tool_choice={"type":"tool","name":"post_review"},
)
import json
review = next(b.input for b in resp.content if b.type == "tool_use")
print(json.dumps(review, indent=2, ensure_ascii=False))

### 3.1 Validación contra schema en el pipeline

In [ ]:
try:
    import jsonschema
    jsonschema.validate(review, REVIEW_TOOL["input_schema"])
    print("OK: review cumple schema, listo para publicar como PR comments")
except ImportError:
    print("(jsonschema no instalado; en producción se valida con jsonschema o pydantic)")
except jsonschema.ValidationError as e:
    print("schema FAIL:", e.message)

## 4. Anti-patrón — review en prosa + grep

In [ ]:
resp_bad = client.messages.create(
    system="Eres un reviewer. Reporta los problemas en formato libre.",
    messages=[{"role":"user","content": f"```diff\n{DIFF}\n```"}],
)
prose = next((b.text for b in resp_bad.content if b.type == "text"), "")
print(prose[:600])

import re
findings = re.findall(r"(line\s*\d+|línea\s*\d+)[:\s]+([^\n]+)", prose, re.IGNORECASE)
print("\nfindings parseados con regex:", findings)

El regex captura algunos hallazgos hoy. Mañana, si el modelo escribe "en la fila 12" o "near line 12" o cambia de idioma, captura cero. El pipeline silenciosamente publica menos comentarios y nadie se entera.

## 5. Argumento de certificación

- **`tool_choice` forzado** evita prosa en la salida.
- **Schema con enums** para `severity`: el pipeline puede mapear a niveles de GitHub Actions sin parsear texto.
- **Validación con `jsonschema`** falla cerrada antes de publicar.
- **Conexión con otros katas**: el reviewer es exactamente la extracción defensiva del Kata 5; corre headless igual que Kata 17 (batches) si querés revisar muchos PRs offline.

## 6. Auto-evaluación

**1. ¿Qué hace el CI si el JSON no valida contra el schema?**

Falla el job, no "ajusta" la salida. Retorno code != 0 y el revisor humano interviene. Es preferible 1 PR sin review automatizado a 100 PRs con reviews silenciosamente vacíos.

**2. ¿Cómo cacheo prompts caros en CI sin invalidar el caché por turno?**

System prompt + reglas estables van con `cache_control: ephemeral` (Kata 10). El diff del PR va al final como `user` content. Mismo prefijo entre PRs distintos pero del mismo proyecto = cache_read alto.

**3. ¿Qué reviews delego al modelo y cuáles dejo para humano?**

Modelo: estilo, secrets, anti-patrones evidentes, lints semánticos. Humano: decisiones arquitectónicas, breaking changes, cambios en contratos públicos. La regla: si requiere conocimiento del negocio, humano.